# (InceptionV3 + EfficientNetB4) + SMOTE + XGBoost — Garlic Classification

**Two-Stage Pipeline:** Fine-tune backbones → extract features → SMOTE → XGBoost

| Item | Detail |
|---|---|
| **Dataset** | 3 classes: Fully_Peeled / Partially_Peeled / Spoiled Garlic |
| **Class distribution (train)** | Fully_Peeled: 1050 · Partially_Peeled: 306 · Spoiled: 704 |
| **Stage 1A** | InceptionV3 fine-tune (mixed7–mixed10) — input 299×299 → 2048-d GAP |
| **Stage 1B** | EfficientNetB4 fine-tune (blocks 3–7) — input 380×380 → 1792-d GAP |
| **Stage 2** | Concatenate → 3840-d · SMOTE (k=5) · XGBoost (GPU) |
| **Why SMOTE?** | Partially_Peeled only 306/2060 ≈ 14.8% → severe minority class |
| **Why InceptionV3?** | Multi-scale Inception modules (1×1, 3×3, 5×5) capture garlic texture at multiple granularities |

---
### InceptionV3 Unfreeze Strategy
```
InceptionV3 has 311 layers organized into Inception blocks: mixed0 → mixed10
Frozen  : mixed0 – mixed6  (low-level: edges, colors, textures)
Trainable: mixed7 – mixed10 (high-level: object parts, complex patterns)
BN layers: always frozen (prevent covariate shift with small dataset)
Trainable params ≈ 14M / 23M total
```

---
### Dataset Class Balance
```
[train]  Fully_Peeled: 1050  |  Partially_Peeled:  306  |  Spoiled:  704  → Total: 2060
[val]    Fully_Peeled:  225  |  Partially_Peeled:   65  |  Spoiled:  150  → Total:  440
[test]   Fully_Peeled:  225  |  Partially_Peeled:   67  |  Spoiled:  152  → Total:  444
GRAND TOTAL: 2944
```
Imbalance ratio ≈ **1050 : 306 : 704** → SMOTE oversamples Partially_Peeled to balance training set.


In [ ]:
# ========== 1. IMPORTS ========== #
import os, csv, time, random, gc
from types import SimpleNamespace

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf

from tensorflow.keras.applications import InceptionV3, EfficientNetB4
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout, BatchNormalization
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, CSVLogger
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.regularizers import l2

from sklearn.utils import class_weight
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, cohen_kappa_score,
    matthews_corrcoef, balanced_accuracy_score,
)
from sklearn.preprocessing import label_binarize
from sklearn.manifold import TSNE
from imblearn.over_sampling import SMOTE
import xgboost as xgb

# ── GPU config ────────────────────────────────────────────────────────────
print("TensorFlow:", tf.__version__)
gpus = tf.config.list_physical_devices('GPU')
print("GPUs:", len(gpus))
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)

# ── Mixed precision ───────────────────────────────────────────────────────
tf.keras.mixed_precision.set_global_policy("mixed_float16")
print("Compute dtype:", tf.keras.mixed_precision.global_policy().compute_dtype)


In [ ]:
# ========== 2. CONFIGURATION ========== #

STRATEGY_KEY   = "inceptionv3_effb4_smote_xgboost"
STRATEGY_LABEL = "(InceptionV3 + EfficientNetB4) Fine-tuned + SMOTE + XGBoost"

# ── Paths ─────────────────────────────────────────────────────────────────
DATA_DIR        = "/kaggle/input/datasets/giaphuc/dataset-garlic-2106/dataset_final_2006"
BASE_RESULT_DIR = f"/kaggle/working/report/{STRATEGY_KEY}"
os.makedirs(BASE_RESULT_DIR, exist_ok=True)

# ── Image sizes ───────────────────────────────────────────────────────────
INCEPTION_SHAPE = (299, 299, 3)   # InceptionV3 native input
EFFB4_SHAPE     = (380, 380, 3)   # EfficientNetB4 native input
BATCH_SIZE      = 32

# ── Fine-tuning ───────────────────────────────────────────────────────────
FT_EPOCHS       = 25
FT_LR           = 1e-4
FT_PATIENCE     = 8
LABEL_SMOOTHING = 0.15
DROPOUT_RATE    = 0.5

# InceptionV3 unfreeze strategy:
#   - 311 total layers, organized into Inception blocks mixed0–mixed10
#   - Freeze mixed0–mixed6 (low-level features)
#   - Unfreeze mixed7–mixed10 (high-level: ~14M / 23M params)
#   - BN layers always frozen to prevent covariate shift
INCEPTION_UNFREEZE_LAYERS = ['mixed7', 'mixed8', 'mixed9', 'mixed10']

# EfficientNetB4: unfreeze blocks 3-4-5-6-7
EFFB4_UNFREEZE_BLOCKS = [3, 4, 5, 6, 7]

# ── SMOTE ─────────────────────────────────────────────────────────────────
SMOTE_K = 5

# ── XGBoost ───────────────────────────────────────────────────────────────
XGB_PARAMS = {
    'objective':        'multi:softprob',
    'eval_metric':      'mlogloss',
    'tree_method':      'hist',
    'device':           'cuda',          # GPU acceleration
    'max_depth':        7,
    'learning_rate':    0.05,
    'n_estimators':     500,
    'subsample':        0.8,
    'colsample_bytree': 0.8,
    'min_child_weight': 3,
    'gamma':            0.1,
    'reg_alpha':        0.1,
    'reg_lambda':       1.0,
    'random_state':     42,
    'verbosity':        1,
}

SEED     = 42
AUTOTUNE = tf.data.AUTOTUNE
tf.config.optimizer.set_jit(True)

all_runs_results = []

# ── Dataset stats (known) ─────────────────────────────────────────────────
print(f"Strategy      : {STRATEGY_LABEL}")
print(f"Dataset       : {DATA_DIR}")
print(f"InceptionV3   : input {INCEPTION_SHAPE} → 2048-d GAP features")
print(f"EfficientNetB4: input {EFFB4_SHAPE} → 1792-d GAP features")
print(f"Fused dim     : {2048 + 1792} = 3840-d")
print(f"SMOTE k       : {SMOTE_K}")
print(f"\nInceptionV3 unfreeze: {INCEPTION_UNFREEZE_LAYERS} (4 of 11 inception blocks)")
print(f"\nDataset distribution:")
print(f"  [train] Fully_Peeled: 1050 | Partially_Peeled: 306 | Spoiled: 704 → 2060")
print(f"  [val]   Fully_Peeled:  225 | Partially_Peeled:  65 | Spoiled: 150 →  440")
print(f"  [test]  Fully_Peeled:  225 | Partially_Peeled:  67 | Spoiled: 152 →  444")
print(f"  Imbalance ratio: 1050 : 306 : 704  (Partially_Peeled is minority ~14.8%)")


In [ ]:
# ========== 3. HELPER FUNCTIONS ========== #

inception_preprocess = tf.keras.applications.inception_v3.preprocess_input
effb4_preprocess     = tf.keras.applications.efficientnet.preprocess_input

_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.083),
    tf.keras.layers.RandomZoom(0.20),
    tf.keras.layers.RandomTranslation(0.20, 0.20),
    tf.keras.layers.RandomBrightness(factor=0.30),
], name='augmentation')


def get_class_info(data_dir):
    class_names  = sorted([
        d for d in os.listdir(os.path.join(data_dir, 'train'))
        if os.path.isdir(os.path.join(data_dir, 'train', d))
    ])
    class_to_idx = {cn: i for i, cn in enumerate(class_names)}
    return class_names, class_to_idx, len(class_names)


def _collect_samples(split_dir, class_to_idx):
    paths, labels, filenames = [], [], []
    for cn, ci in sorted(class_to_idx.items()):
        d = os.path.join(split_dir, cn)
        for fname in sorted(os.listdir(d)):
            if fname.lower().endswith(('.jpg','.jpeg','.png','.bmp','.tiff')):
                paths.append(os.path.join(d, fname))
                labels.append(ci)
                filenames.append(f"{cn}/{fname}")
    return paths, labels, filenames


def create_tf_datasets(data_dir, input_shape, batch_size, preprocess_fn,
                       seed=None, use_aug=True):
    class_names, class_to_idx, num_classes = get_class_info(data_dir)
    h, w = input_shape[:2]

    def load_and_preprocess(path, label):
        img   = tf.image.decode_jpeg(tf.io.read_file(path), channels=3)
        img   = tf.image.resize(img, [h, w])
        img   = preprocess_fn(tf.cast(img, tf.float32))
        label = tf.one_hot(label, depth=num_classes)
        return img, label

    def augment(img, lbl):
        return _augmentation(img, training=True), lbl

    def _make_split(split, training=False, apply_aug=False):
        sdir = os.path.join(data_dir, split)
        paths, labels, fns = _collect_samples(sdir, class_to_idx)
        n  = len(paths)
        ds = tf.data.Dataset.from_tensor_slices((paths, labels))
        if training:
            ds = ds.shuffle(n, seed=seed, reshuffle_each_iteration=True)
        ds = ds.map(load_and_preprocess, num_parallel_calls=AUTOTUNE)
        if apply_aug:
            ds = ds.map(augment, num_parallel_calls=AUTOTUNE)
        ds = ds.batch(batch_size, drop_remainder=training).prefetch(AUTOTUNE)
        return ds, n, fns, labels

    train_ds, n_train, _,           train_lbl = _make_split('train', True,  use_aug)
    val_ds,   n_val,   _,           _         = _make_split('val',   False, False)
    test_ds,  n_test,  test_fnames, test_lbl  = _make_split('test',  False, False)

    cw_dict = dict(enumerate(
        class_weight.compute_class_weight(
            'balanced', classes=np.unique(train_lbl), y=train_lbl)))

    meta = SimpleNamespace(
        class_names=class_names, class_to_idx=class_to_idx,
        num_classes=num_classes, test_filenames=test_fnames,
        test_classes=np.array(test_lbl),
        n_train=n_train, n_val=n_val, n_test=n_test,
        class_weight_dict=cw_dict,
    )
    print(f"  train={n_train}  val={n_val}  test={n_test}")
    return train_ds, val_ds, test_ds, meta


def build_finetune_model(backbone, num_classes, name):
    x   = GlobalAveragePooling2D(name=f'{name}_gap')(backbone.output)
    x   = BatchNormalization(name=f'{name}_head_bn')(x)
    x   = Dense(256, activation='relu',
                kernel_regularizer=l2(1e-5), name=f'{name}_head_dense')(x)
    x   = Dropout(DROPOUT_RATE, name=f'{name}_head_dropout')(x)
    out = Dense(num_classes, activation='softmax',
                dtype='float32', name=f'{name}_predictions')(x)
    return Model(inputs=backbone.input, outputs=out, name=f'{name}_finetune')


def extract_gap_features(feat_model, data_dir, split, class_to_idx,
                         input_shape, preprocess_fn):
    """Extract GAP features batch-by-batch (memory safe)."""
    sdir = os.path.join(data_dir, split)
    paths, labels, fnames = _collect_samples(sdir, class_to_idx)
    h, w = input_shape[:2]
    feats = []
    for start in range(0, len(paths), BATCH_SIZE):
        batch_paths = paths[start:start+BATCH_SIZE]
        batch = np.stack([
            preprocess_fn(tf.cast(
                tf.image.resize(
                    tf.image.decode_jpeg(tf.io.read_file(p), channels=3),
                    [h, w]), tf.float32).numpy())
            for p in batch_paths
        ])
        feats.append(feat_model(batch, training=False).numpy().astype(np.float32))
        print(f"    {split}: {min(start+BATCH_SIZE, len(paths))}/{len(paths)}", end='\r')
    feats = np.concatenate(feats)
    print(f"  {split}: {len(paths)} imgs → {feats.shape}")
    return feats, np.array(labels), fnames


print("Helper functions defined.")
print(f"  InceptionV3 GAP output   : 2048-d")
print(f"  EfficientNetB4 GAP output: 1792-d")
print(f"  Fused feature vector     : 3840-d")


In [ ]:
# ========== 4. STAGE 1A — FINE-TUNE InceptionV3 ========== #

random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

class_names, class_to_idx, num_classes = get_class_info(DATA_DIR)
print(f"Classes ({num_classes}): {class_names}")

RESULT_DIR = os.path.join(BASE_RESULT_DIR, f"run_1_seed_{SEED}")
os.makedirs(RESULT_DIR, exist_ok=True)

print("\n[Stage 1A] Fine-tuning InceptionV3 ...")
train_ds_i, val_ds_i, _, meta = create_tf_datasets(
    DATA_DIR, INCEPTION_SHAPE, BATCH_SIZE, inception_preprocess, seed=SEED, use_aug=True)

# Build InceptionV3: freeze all → unfreeze mixed7–mixed10 (BN always frozen)
inception_base = InceptionV3(weights='imagenet', include_top=False,
                             input_shape=INCEPTION_SHAPE)
inception_base.trainable = False

for layer in inception_base.layers:
    # Unfreeze layers whose name starts with any of the target mixed blocks
    if any(layer.name.startswith(block) for block in INCEPTION_UNFREEZE_LAYERS):
        if not isinstance(layer, tf.keras.layers.BatchNormalization):
            layer.trainable = True

trainable_count = sum(1 for l in inception_base.layers if l.trainable)
print(f"  Unfreeze blocks     : {INCEPTION_UNFREEZE_LAYERS}")
print(f"  Trainable backbone layers: {trainable_count}/{len(inception_base.layers)}")

# Verify which blocks are unfrozen
unfrozen_blocks = set()
for layer in inception_base.layers:
    if layer.trainable:
        for block in INCEPTION_UNFREEZE_LAYERS:
            if layer.name.startswith(block):
                unfrozen_blocks.add(block)
print(f"  Unfrozen block names: {sorted(unfrozen_blocks)}")

inception_model = build_finetune_model(inception_base, num_classes, 'inception')
steps_i = meta.n_train // BATCH_SIZE

inception_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        tf.keras.optimizers.schedules.ExponentialDecay(
            FT_LR, decay_steps=steps_i*5, decay_rate=0.9, staircase=True)),
    loss=CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy'])

print(f"  Trainable params: {sum(tf.size(v).numpy() for v in inception_model.trainable_weights):,}")

history_i = inception_model.fit(
    train_ds_i, validation_data=val_ds_i,
    epochs=FT_EPOCHS,
    class_weight=meta.class_weight_dict,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=FT_PATIENCE,
                      restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'inceptionv3_log.csv')),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'inceptionv3_best.keras'),
                        save_best_only=True, monitor='val_loss', verbose=1),
    ],
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (k1,k2), title in zip(axes,
    [('accuracy','val_accuracy'),('loss','val_loss')], ['Accuracy','Loss']):
    ax.plot(history_i.history[k1], label='Train')
    ax.plot(history_i.history[k2], label='Val')
    ax.set_title(f'InceptionV3 {title}'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'inceptionv3_curves.png'), dpi=300)
plt.show()
print("InceptionV3 fine-tuning complete.")


In [ ]:
# ========== 5. STAGE 1B — FINE-TUNE EfficientNetB4 ========== #

gc.collect(); tf.keras.backend.clear_session()
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("[Stage 1B] Fine-tuning EfficientNetB4 ...")
train_ds_e, val_ds_e, _, meta = create_tf_datasets(
    DATA_DIR, EFFB4_SHAPE, BATCH_SIZE, effb4_preprocess, seed=SEED, use_aug=True)

# Build EfficientNetB4: freeze all → unfreeze blocks 3-7 (BN always frozen)
effb4_base = EfficientNetB4(weights='imagenet', include_top=False, input_shape=EFFB4_SHAPE)
effb4_base.trainable = False
for layer in effb4_base.layers:
    for bn in EFFB4_UNFREEZE_BLOCKS:
        if layer.name.startswith(f"block{bn}"):
            if not isinstance(layer, tf.keras.layers.BatchNormalization):
                layer.trainable = True
            break

print(f"  Trainable backbone layers: "
      f"{sum(1 for l in effb4_base.layers if l.trainable)}/{len(effb4_base.layers)}")

effb4_model = build_finetune_model(effb4_base, num_classes, 'effb4')
steps_e = meta.n_train // BATCH_SIZE

effb4_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        tf.keras.optimizers.schedules.ExponentialDecay(
            FT_LR, decay_steps=steps_e*5, decay_rate=0.9, staircase=True)),
    loss=CategoricalCrossentropy(label_smoothing=LABEL_SMOOTHING),
    metrics=['accuracy'])

print(f"  Trainable params: {sum(tf.size(v).numpy() for v in effb4_model.trainable_weights):,}")

history_e = effb4_model.fit(
    train_ds_e, validation_data=val_ds_e,
    epochs=FT_EPOCHS,
    class_weight=meta.class_weight_dict,
    callbacks=[
        EarlyStopping(monitor='val_loss', patience=FT_PATIENCE,
                      restore_best_weights=True, verbose=1),
        CSVLogger(os.path.join(RESULT_DIR, 'effb4_log.csv')),
        ModelCheckpoint(os.path.join(RESULT_DIR, 'effb4_best.keras'),
                        save_best_only=True, monitor='val_loss', verbose=1),
    ],
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (k1,k2), title in zip(axes,
    [('accuracy','val_accuracy'),('loss','val_loss')], ['Accuracy','Loss']):
    ax.plot(history_e.history[k1], label='Train')
    ax.plot(history_e.history[k2], label='Val')
    ax.set_title(f'EfficientNetB4 {title}'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'effb4_curves.png'), dpi=300)
plt.show()
print("EfficientNetB4 fine-tuning complete.")


In [ ]:
# ========== 6. STAGE 2A — EXTRACT GAP FEATURES ========== #

print("[Stage 2A] Extracting GAP feature vectors ...")

# --- Load best fine-tuned models ---
inception_model = tf.keras.models.load_model(os.path.join(RESULT_DIR, 'inceptionv3_best.keras'))
effb4_model     = tf.keras.models.load_model(os.path.join(RESULT_DIR, 'effb4_best.keras'))

# Build GAP-only extractors (output = penultimate vector before classification head)
inception_extractor = tf.keras.Model(
    inputs=inception_model.input,
    outputs=inception_model.get_layer('inception_gap').output,
    name='inception_extractor')
effb4_extractor = tf.keras.Model(
    inputs=effb4_model.input,
    outputs=effb4_model.get_layer('effb4_gap').output,
    name='effb4_extractor')

print(f"  InceptionV3 GAP dim   : {inception_extractor.output_shape[-1]}")
print(f"  EfficientNetB4 GAP dim: {effb4_extractor.output_shape[-1]}")
print(f"  Fused dim             : {inception_extractor.output_shape[-1] + effb4_extractor.output_shape[-1]}")

# Build datasets (no augmentation for feature extraction)
_, class_to_idx, _ = get_class_info(DATA_DIR)

for split_name in ['train', 'val', 'test']:
    feats_i, labels_i, _ = extract_gap_features(
        inception_extractor, DATA_DIR, split_name, class_to_idx,
        INCEPTION_SHAPE, inception_preprocess)
    feats_e, labels_e, _ = extract_gap_features(
        effb4_extractor, DATA_DIR, split_name, class_to_idx,
        EFFB4_SHAPE, effb4_preprocess)

    assert np.array_equal(labels_i, labels_e), "Label mismatch between datasets!"

    fused = np.concatenate([feats_i, feats_e], axis=1)
    np.save(os.path.join(RESULT_DIR, f'X_{split_name}.npy'), fused)
    np.save(os.path.join(RESULT_DIR, f'y_{split_name}.npy'), labels_i)
    print(f"  [{split_name}] X={fused.shape}  y={labels_i.shape}")

class_names, _, _ = get_class_info(DATA_DIR)
print("Feature extraction complete.")


In [ ]:
# ========== 7. STAGE 2B — SMOTE OVERSAMPLING ========== #

print("[Stage 2B] SMOTE oversampling ...")

X_train = np.load(os.path.join(RESULT_DIR, 'X_train.npy'))
y_train = np.load(os.path.join(RESULT_DIR, 'y_train.npy'))
X_val   = np.load(os.path.join(RESULT_DIR, 'X_val.npy'))
y_val   = np.load(os.path.join(RESULT_DIR, 'y_val.npy'))
X_test  = np.load(os.path.join(RESULT_DIR, 'X_test.npy'))
y_test  = np.load(os.path.join(RESULT_DIR, 'y_test.npy'))

print("  Before SMOTE:")
for i, name in enumerate(class_names):
    print(f"    {name}: {(y_train==i).sum()}")

smote = SMOTE(k_neighbors=SMOTE_K, random_state=SEED)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("\n  After SMOTE:")
for i, name in enumerate(class_names):
    print(f"    {name}: {(y_train_sm==i).sum()}")

# Visualise class distribution before / after
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, (X_, y_, title) in zip(axes, [
        (X_train,    y_train,    'Before SMOTE'),
        (X_train_sm, y_train_sm, 'After SMOTE')]):
    counts = [int((y_==i).sum()) for i in range(len(class_names))]
    bars = ax.bar(class_names, counts, color=['#4C72B0','#DD8452','#55A868'])
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_ylabel('Samples')
    for bar, cnt in zip(bars, counts):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+5,
                str(cnt), ha='center', va='bottom', fontsize=10)
    ax.set_ylim(0, max(counts)*1.2)
    ax.tick_params(axis='x', rotation=15)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'smote_distribution.png'), dpi=300)
plt.show()
print("SMOTE oversampling complete.")


In [ ]:
# ========== 8. STAGE 3 — XGBoost TRAINING ========== #

print("[Stage 3] Training XGBoost classifier ...")

xgb_clf = xgb.XGBClassifier(
    **XGB_PARAMS,
    num_class=len(class_names),
    use_label_encoder=False,
    eval_metric=['mlogloss', 'merror'],
    random_state=SEED,
)

xgb_clf.fit(
    X_train_sm, y_train_sm,
    eval_set=[(X_train_sm, y_train_sm), (X_val, y_val)],
    early_stopping_rounds=30,
    verbose=50,
)

# Save model
xgb_path = os.path.join(RESULT_DIR, 'xgboost_model.json')
xgb_clf.save_model(xgb_path)
print(f"\n  Best iteration : {xgb_clf.best_iteration}")
print(f"  Best val logloss: {xgb_clf.best_score:.4f}")
print(f"  Model saved to : {xgb_path}")

# Plot training curves
evals = xgb_clf.evals_result()
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, key, title in zip(axes, ['mlogloss','merror'], ['Log-Loss','Error Rate']):
    ax.plot(evals['validation_0'][key], label='Train', alpha=0.8)
    ax.plot(evals['validation_1'][key], label='Val',   alpha=0.8)
    ax.axvline(xgb_clf.best_iteration, color='red', linestyle='--',
               alpha=0.6, label=f'Best iter={xgb_clf.best_iteration}')
    ax.set_title(f'XGBoost {title}'); ax.legend(); ax.grid(alpha=0.3)
    ax.set_xlabel('Boosting Round')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'xgb_curves.png'), dpi=300)
plt.show()
print("XGBoost training complete.")


In [ ]:
# ========== 9. EVALUATION ========== #

from sklearn.metrics import (classification_report, confusion_matrix,
                              roc_auc_score, cohen_kappa_score,
                              matthews_corrcoef, balanced_accuracy_score,
                              roc_curve, auc)

print("[Evaluation] Running on Val + Test splits ...")
results = {}

for split_name, X_, y_ in [('val', X_val, y_val), ('test', X_test, y_test)]:
    y_pred  = xgb_clf.predict(X_)
    y_proba = xgb_clf.predict_proba(X_)

    acc  = (y_pred == y_).mean()
    kap  = cohen_kappa_score(y_, y_pred)
    mcc  = matthews_corrcoef(y_, y_pred)
    bacc = balanced_accuracy_score(y_, y_pred)
    try:
        auc_ovo = roc_auc_score(y_, y_proba, multi_class='ovo', average='macro')
    except Exception:
        auc_ovo = float('nan')

    results[split_name] = dict(acc=acc, kappa=kap, mcc=mcc,
                                balanced_acc=bacc, auc_ovo=auc_ovo)

    print(f"\n  ===== {split_name.upper()} SET =====")
    print(f"  Accuracy        : {acc:.4f}")
    print(f"  Balanced Acc    : {bacc:.4f}")
    print(f"  Cohen Kappa     : {kap:.4f}")
    print(f"  MCC             : {mcc:.4f}")
    print(f"  AUC-ROC (OvO)   : {auc_ovo:.4f}")
    print()
    print(classification_report(y_, y_pred, target_names=class_names, digits=4))

# ── Confusion Matrices ──────────────────────────────────────────────────── #
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, split_name, X_, y_ in [
        (axes[0], 'Val',  X_val,  y_val),
        (axes[1], 'Test', X_test, y_test)]:
    y_pred = xgb_clf.predict(X_)
    cm = confusion_matrix(y_, y_pred)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    im = ax.imshow(cm_norm, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(len(class_names))); ax.set_xticklabels(class_names, rotation=20, ha='right')
    ax.set_yticks(range(len(class_names))); ax.set_yticklabels(class_names)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    ax.set_title(f'Confusion Matrix — {split_name}', fontweight='bold')
    for i in range(len(class_names)):
        for j in range(len(class_names)):
            ax.text(j, i, f'{cm[i,j]}\n({cm_norm[i,j]:.2%})',
                    ha='center', va='center',
                    color='white' if cm_norm[i,j] > 0.5 else 'black', fontsize=9)
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'confusion_matrices.png'), dpi=300)
plt.show()

# ── ROC Curves (Test) ───────────────────────────────────────────────────── #
y_proba_test = xgb_clf.predict_proba(X_test)
fig, ax = plt.subplots(figsize=(7, 6))
colors = ['#4C72B0', '#DD8452', '#55A868']
for i, (name, color) in enumerate(zip(class_names, colors)):
    y_bin = (y_test == i).astype(int)
    fpr, tpr, _ = roc_curve(y_bin, y_proba_test[:, i])
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, lw=2, label=f'{name} (AUC={roc_auc:.4f})')
ax.plot([0,1],[0,1],'k--',lw=1,alpha=0.5)
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves (Test Set)', fontweight='bold')
ax.legend(loc='lower right'); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'roc_curves.png'), dpi=300)
plt.show()
print("Evaluation complete.")


In [ ]:
# ========== 10. VISUALISATIONS — Feature Importance & t-SNE ========== #

from sklearn.manifold import TSNE
from matplotlib.patches import Patch

# ── Feature Importance (Top-20) ─────────────────────────────────────────── #
importances = xgb_clf.feature_importances_         # shape: (3840,)
top_k = 20
top_idx = np.argsort(importances)[::-1][:top_k]

# Colour by backbone: 0-2047 → InceptionV3 (blue), 2048-3839 → EfficientNetB4 (orange)
colors_bar = ['#4C72B0' if i < 2048 else '#DD8452' for i in top_idx]
labels_bar = [f'{"I" if i < 2048 else "E"}{i if i < 2048 else i-2048}' for i in top_idx]

fig, ax = plt.subplots(figsize=(12, 5))
bars = ax.bar(range(top_k), importances[top_idx], color=colors_bar)
ax.set_xticks(range(top_k)); ax.set_xticklabels(labels_bar, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('F-score Importance'); ax.set_title('Top-20 XGBoost Feature Importances', fontweight='bold')
legend_patches = [Patch(color='#4C72B0', label='InceptionV3 (dim 0–2047)'),
                  Patch(color='#DD8452', label='EfficientNetB4 (dim 2048–3839)')]
ax.legend(handles=legend_patches)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'feature_importance.png'), dpi=300)
plt.show()

# Backbone contribution (% of top-200 features)
top200 = np.argsort(importances)[::-1][:200]
pct_inception = (top200 < 2048).mean() * 100
print(f"  InceptionV3 contributes    {pct_inception:.1f}% of top-200 features")
print(f"  EfficientNetB4 contributes {100-pct_inception:.1f}% of top-200 features")

# ── t-SNE: Before & After SMOTE ─────────────────────────────────────────── #
print("\n  Running t-SNE (this may take ~1 min) ...")
palette = ['#4C72B0','#DD8452','#55A868']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, (X_, y_, title) in zip(axes, [
        (X_train,    y_train,    'Before SMOTE (train)'),
        (X_train_sm, y_train_sm, 'After  SMOTE (train)')]):

    max_pts = 2000
    if len(X_) > max_pts:
        idx = np.random.choice(len(X_), max_pts, replace=False)
        X_s, y_s = X_[idx], y_[idx]
    else:
        X_s, y_s = X_, y_

    emb = TSNE(n_components=2, random_state=SEED, perplexity=30,
               n_iter=1000).fit_transform(X_s)
    for i, (name, color) in enumerate(zip(class_names, palette)):
        mask = y_s == i
        ax.scatter(emb[mask, 0], emb[mask, 1], c=color,
                   label=f'{name} (n={mask.sum()})', alpha=0.6, s=15, edgecolors='none')
    ax.set_title(title, fontweight='bold'); ax.legend(fontsize=8)
    ax.set_xticks([]); ax.set_yticks([])
plt.suptitle('t-SNE of Fused GAP Features (InceptionV3 + EfficientNetB4)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(os.path.join(RESULT_DIR, 'tsne_smote.png'), dpi=300)
plt.show()
print("Visualisation complete.")


In [ ]:
# ========== 11. FINAL REPORT & EXPORT ========== #

import zipfile, json, datetime

print("[Final Report] Compiling results ...")

# ── Metrics table ───────────────────────────────────────────────────────── #
rows = []
for split_name, X_, y_ in [('val', X_val, y_val), ('test', X_test, y_test)]:
    y_pred  = xgb_clf.predict(X_)
    y_proba = xgb_clf.predict_proba(X_)
    from sklearn.metrics import f1_score, precision_score, recall_score
    rows.append({
        'split'          : split_name,
        'accuracy'       : f"{(y_pred==y_).mean():.4f}",
        'balanced_acc'   : f"{balanced_accuracy_score(y_, y_pred):.4f}",
        'precision_macro': f"{precision_score(y_, y_pred, average='macro', zero_division=0):.4f}",
        'recall_macro'   : f"{recall_score(y_, y_pred, average='macro', zero_division=0):.4f}",
        'f1_macro'       : f"{f1_score(y_, y_pred, average='macro', zero_division=0):.4f}",
        'f1_weighted'    : f"{f1_score(y_, y_pred, average='weighted', zero_division=0):.4f}",
        'cohen_kappa'    : f"{cohen_kappa_score(y_, y_pred):.4f}",
        'mcc'            : f"{matthews_corrcoef(y_, y_pred):.4f}",
    })

df_results = pd.DataFrame(rows)
print("\n", df_results.to_string(index=False))
df_results.to_csv(os.path.join(RESULT_DIR, 'final_report.csv'), index=False)

# ── Strategy summary ─────────────────────────────────────────────────────── #
strategy = {
    'timestamp'              : datetime.datetime.now().isoformat(),
    'backbone_1'             : 'InceptionV3 (ImageNet, GAP=2048, unfreeze=mixed7-mixed10)',
    'backbone_2'             : 'EfficientNetB4 (ImageNet, GAP=1792, unfreeze=blocks3-7)',
    'inception_unfreeze'     : INCEPTION_UNFREEZE_LAYERS,
    'effb4_unfreeze_blocks'  : EFFB4_UNFREEZE_BLOCKS,
    'fused_dim'              : 3840,
    'smote_k'                : SMOTE_K,
    'xgb_params'             : XGB_PARAMS,
    'xgb_best_iteration'     : int(xgb_clf.best_iteration),
    'dataset_train_orig'     : {cn: int((y_train==i).sum()) for i,cn in enumerate(class_names)},
    'dataset_train_smote'    : {cn: int((y_train_sm==i).sum()) for i,cn in enumerate(class_names)},
    'val_metrics'            : results.get('val', {}),
    'test_metrics'           : results.get('test', {}),
}
with open(os.path.join(RESULT_DIR, 'strategy_summary.json'), 'w') as f:
    json.dump(strategy, f, indent=2, default=str)

# ── Zip everything ────────────────────────────────────────────────────────── #
zip_path = os.path.join(RESULT_DIR, 'inceptionv3_effb4_smote_xgb_results.zip')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in os.listdir(RESULT_DIR):
        if fn != os.path.basename(zip_path):
            zf.write(os.path.join(RESULT_DIR, fn), fn)

print(f"\n  All artifacts saved to : {RESULT_DIR}")
print(f"  ZIP archive            : {zip_path}")
print("\n====== Pipeline Complete ======")
print(df_results.to_string(index=False))
